In [28]:
!pip install emoji
import pandas as pd
import emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 5.6 MB/s  0:00:00


In [2]:
TRAIN_PATH = 'INPUT/train.csv'
TEST_PATH  = 'INPUT/test.csv'
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

In [4]:
df1 = pd.DataFrame({
    "body": df_train["positive_example_1"],
    "rule_violation": 1
})

df2 = pd.DataFrame({
    "body": df_train["positive_example_2"],
    "rule_violation": 1
})

df3 = pd.DataFrame({
    "body": df_train["negative_example_1"],
    "rule_violation": 0
})

df4 = pd.DataFrame({
    "body": df_train["negative_example_2"],
    "rule_violation": 0
})
df = pd.concat([df1 , df2, df3, df4], ignore_index=True)

In [5]:
df.head()

,body,rule_violation
0,If you could tell your younger self something ...,1
1,[I wanna kiss you all over! Stunning!](http://...,1
2,Don't break up with him or call the cops. If ...,1
3,Selling Tyrande codes for 3€ to paypal. PM. \n...,1
4,wow!! amazing reminds me of the old days.Well...,1


In [6]:
df.shape

(8116, 2)

In [8]:
import spacy
nlp = spacy.load("en_core_web_lg")

In [19]:
#def get_features(df, column_name):    
#    df[new_column_name] = df[column_name].apply(
#        lambda text: len([token.text for token in nlp(str(text)) if token.is_alpha])
#    )
#    return df
import re
import numpy as np
def get_features(text):
    one_line_text = text.replace('\n', ' ')
    one_line_text = re.sub(r'\s+', ' ', one_line_text).strip()
    
    doc = nlp(one_line_text)

    features = {
        'body_length': sum(len(token.text) for token in doc if not token.is_space),
        'word_counts': len([token.text for token in doc]),
        'alpha_word_counts': len([token.text for token in doc if token.is_alpha]),
        'xx': len(" ".join([token.lemma_ for token in nlp(str(text)) if not token.is_punct])),
    }
    
    return features

features_df = df_train['body'].apply(get_features).apply(pd.Series)
features_df.head()
#df = pd.concat([df, features_df], axis=1)

,body_length,word_counts,alpha_word_counts,xx
0,48,15,12,57
1,85,10,4,86
2,46,15,12,55
3,62,12,11,76
4,288,33,16,310


In [35]:
import emoji
def get_features(text):
    text = " ".join([token.lemma_ for token in nlp(str(text)) ])
    
    doc = nlp(text)

    http_urls = []
    https_urls = []

    for token in doc:
        if token.like_url:
            if token.text.startswith('http://'):
                http_urls.append(token.text)
            elif token.text.startswith('https://'):
                https_urls.append(token.text)
    
    # Phone number regex (simplified)
    phone_pattern = r'(\+?\d{1,3}[-.\s]?)?(\(?\d{1,4}\)?[-.\s]?)?[\d\s.-]{5,}'
    phone_matches = re.findall(phone_pattern, text)

    # Count emojis (character-level)
    emoji_count = sum(1 for char in text if emoji.is_emoji(char))
    
    features = {
        'body_length': sum(len(token.text) for token in doc if not token.is_space),
        'word_counts': len([token.text for token in doc]),
        'alpha_word_counts': len([token.text for token in doc if token.is_alpha]),
        'punctuation_counts': len([token.text for token in doc if token.is_punct]),
        'email_counts': len([token.text for token in doc if token.like_email]),
        'phone_counts': len(phone_matches),
        'http_counts': len(http_urls),
        'https_counts': len(https_urls),
        'url_length': sum(len(item) for item in http_urls + https_urls),
        'url_digits': sum(char.isdigit() for item in http_urls + https_urls for char in item),
        'emoji_counts': emoji_count,
    }
    
    return features

features_df = df_train['body'].apply(get_features).apply(pd.Series)
features_df.head()

,body_length,word_counts,alpha_word_counts,punctuation_counts,email_counts,phone_counts,http_counts,https_counts,url_length,url_digits,emoji_counts
0,47,15,13,2,0,0,0,0,0,0,0
1,85,10,4,4,0,0,1,0,65,3,0
2,45,15,13,2,0,0,0,0,0,0,0
3,61,14,11,0,0,0,1,0,22,1,0
4,286,36,16,6,0,3,0,2,164,8,0


In [52]:
def get_features(text):
    text = " ".join([token.lemma_ for token in nlp(str(text)) ])
    
    doc = nlp(text)
    print (doc)
    http_urls = []
    https_urls = []

    for token in doc:
        if token.like_url:
            print (token)
            if token.text.startswith('http://'):
                http_urls.append(token.text)
            elif token.text.startswith('https://'):
                https_urls.append(token.text)
    print (http_urls)
    # Phone number regex (simplified)
    phone_pattern = r'(\+?\d{1,3}[-.\s]?)?(\(?\d{1,4}\)?[-.\s]?)?[\d\s.-]{5,}'
    phone_matches = re.findall(phone_pattern, text)

    # Count emojis (character-level)
    emoji_count = sum(1 for char in text if emoji.is_emoji(char))
    
    features = {
        'body_length': sum(len(token.text) for token in doc if not token.is_space),
        'word_counts': len([token.text for token in doc]),
        'alpha_word_counts': len([token.text for token in doc if token.is_alpha]),
        'punctuation_counts': len([token.text for token in doc if token.is_punct]),
        'email_counts': len([token.text for token in doc if token.like_email]),
        'phone_counts': len(phone_matches),
        'http_counts': len(http_urls),
        'https_counts': len(https_urls),
        'url_length': sum(len(item) for item in http_urls + https_urls),
        'url_digits': sum(char.isdigit() for item in http_urls + https_urls for char in item),
        'url_params': sum(item.count("&") for item in http_urls + https_urls),
        'emoji_counts': emoji_count,
    }
    
    return features

cell_str = str(df_train.loc[4, 'body'])
#print (cell_str)
features = get_features(cell_str)
print (features)

code free tyrande --- > > > [ imgur](http://i.imgur.com / klvsscl.png ) 

 for you and your friend 2 code for 4 dollar https://www.paypal.com/cgi-bin/webscr?cmd=_s-xclick&hosted_button_id=un4e27ag7bwks 

 2 $ ... buy one directly from here : https://www.paypal.com/cgi-bin/webscr?cmd=_s-xclick&hosted_button_id=vp3s5hqre7t7e 

imgur](http://i.imgur.com
klvsscl.png
https://www.paypal.com/cgi-bin/webscr?cmd=_s-xclick&hosted_button_id=un4e27ag7bwks
https://www.paypal.com/cgi-bin/webscr?cmd=_s-xclick&hosted_button_id=vp3s5hqre7t7e
[]
{'body_length': 286, 'word_counts': 36, 'alpha_word_counts': 16, 'punctuation_counts': 6, 'email_counts': 0, 'phone_counts': 3, 'http_counts': 0, 'https_counts': 2, 'url_length': 164, 'url_digits': 8, 'url_params': 2, 'emoji_counts': 0}


In [56]:
def get_features(text):
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    print (text)
    print ("\n\n")
    text = " ".join([token.lemma_ for token in nlp(str(text)) ])
    features = text
    return features
cell_str = str(df_train.loc[4, 'body'])
#print (cell_str)
features = get_features(cell_str)
print (features)

code free tyrande --->>> [Imgur](http://i.imgur.com/KlvssCl.png) for you and your friend 2 codes for 4 dollars https://www.paypal.com/cgi-bin/webscr?cmd=_s-xclick&hosted_button_id=UN4E27AG7BWKS 2$... buy one directly from here: https://www.paypal.com/cgi-bin/webscr?cmd=_s-xclick&hosted_button_id=VP3S5HQRE7T7E



code free tyrande --- > > > [ imgur](http://i.imgur.com / klvsscl.png ) for you and your friend 2 code for 4 dollar https://www.paypal.com/cgi-bin/webscr?cmd=_s-xclick&hosted_button_id=un4e27ag7bwks 2 $ ... buy one directly from here : https://www.paypal.com/cgi-bin/webscr?cmd=_s-xclick&hosted_button_id=vp3s5hqre7t7e
